# Eliminar outliers (sale y rent)

Reglas aplicadas sobre copias:
1. Umbral fijo con OR:
   - SALE: `precio > 1_400_000` **o** `superficie_construida_m2 > 750`
   - RENT: `precio > 3900` **o** `superficie_construida_m2 > 300`
2. Filtro adicional iterativo de `3σ` sobre **precio** y **superficie_construida_m2**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")

def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data" / "processed" / "idealistaAPI").exists():
            return candidate
    raise FileNotFoundError("No se encontro la raiz del proyecto con data/processed/idealistaAPI")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "idealistaAPI"

SALE_PATH = DATA_DIR / "sale_homes_clean.csv"
RENT_PATH = DATA_DIR / "rent_homes_clean.csv"

SALE_OUT_PATH = DATA_DIR / "sale_homes_clean_outliers.csv"
RENT_OUT_PATH = DATA_DIR / "rent_homes_clean_outliers.csv"

sale_df = pd.read_csv(SALE_PATH)
rent_df = pd.read_csv(RENT_PATH)

print(f"SALE original: {sale_df.shape}")
print(f"RENT original: {rent_df.shape}")

In [ ]:
def remove_sigma_iterative(df: pd.DataFrame, sigma: float = 3.0):
    out = df.copy()
    removed_idx = set()
    removed_total = 0
    iterations = 0

    while True:
        iterations += 1

        mu_p = out["precio"].mean()
        sd_p = out["precio"].std(ddof=0)
        mu_s = out["superficie_construida_m2"].mean()
        sd_s = out["superficie_construida_m2"].std(ddof=0)

        if pd.isna(sd_p) or sd_p == 0:
            mask_p = pd.Series(False, index=out.index)
        else:
            mask_p = ((out["precio"] - mu_p).abs() / sd_p) > sigma

        if pd.isna(sd_s) or sd_s == 0:
            mask_s = pd.Series(False, index=out.index)
        else:
            mask_s = ((out["superficie_construida_m2"] - mu_s).abs() / sd_s) > sigma

        mask = mask_p | mask_s
        n = int(mask.sum())

        if n == 0:
            break

        removed_total += n
        removed_idx.update(out.index[mask].tolist())
        out = out.loc[~mask].copy()

    return out, removed_total, removed_idx, iterations - 1


def clean_dataset(df: pd.DataFrame, price_thr: float, surf_thr: float, label: str):
    original = df.copy()

    # Regla fija (OR)
    fixed_mask = (original["precio"] > price_thr) | (original["superficie_construida_m2"] > surf_thr)
    fixed_idx = set(original.index[fixed_mask].tolist())
    after_fixed = original.loc[~fixed_mask].copy()

    # Estadisticas 3 sigma iniciales (sobre precio y superficie)
    mu_p0 = after_fixed["precio"].mean()
    sd_p0 = after_fixed["precio"].std(ddof=0)
    mu_s0 = after_fixed["superficie_construida_m2"].mean()
    sd_s0 = after_fixed["superficie_construida_m2"].std(ddof=0)

    # 3 sigma iterativo
    clean, sigma_removed, sigma_idx, n_iters = remove_sigma_iterative(after_fixed, sigma=3.0)

    # Verificaciones finales
    fixed_left = int(((clean["precio"] > price_thr) | (clean["superficie_construida_m2"] > surf_thr)).sum())

    mu_pf = clean["precio"].mean()
    sd_pf = clean["precio"].std(ddof=0)
    mu_sf = clean["superficie_construida_m2"].mean()
    sd_sf = clean["superficie_construida_m2"].std(ddof=0)

    sigma_left_p = 0 if (pd.isna(sd_pf) or sd_pf == 0) else int((((clean["precio"] - mu_pf).abs() / sd_pf) > 3).sum())
    sigma_left_s = 0 if (pd.isna(sd_sf) or sd_sf == 0) else int((((clean["superficie_construida_m2"] - mu_sf).abs() / sd_sf) > 3).sum())

    print(f"\n===== {label} =====")
    print(f"Eliminados por umbral fijo (OR): {len(fixed_idx)}")
    print(f"Eliminados por 3 sigma: {sigma_removed}")
    print(f"Iteraciones 3 sigma: {n_iters}")
    print(f"Filas finales: {clean.shape[0]}")
    print(f"Restantes fuera del umbral fijo: {fixed_left}")
    print(f"Restantes >3 sigma precio: {sigma_left_p}")
    print(f"Restantes >3 sigma superficie: {sigma_left_s}")

    return {
        "original": original,
        "after_fixed": after_fixed,
        "clean": clean,
        "fixed_idx": fixed_idx,
        "sigma_idx": sigma_idx,
        "mu_p0": mu_p0,
        "sd_p0": sd_p0,
        "mu_s0": mu_s0,
        "sd_s0": sd_s0,
    }


sale_res = clean_dataset(sale_df, price_thr=1_400_000, surf_thr=750, label="SALE")
rent_res = clean_dataset(rent_df, price_thr=3_900, surf_thr=300, label="RENT")

In [ ]:
def plot_outliers_clear(result: dict, label: str, price_thr: float, surf_thr: float):
    original = result["original"]
    after_fixed = result["after_fixed"]
    clean = result["clean"]

    fixed_idx = result["fixed_idx"]
    sigma_idx = result["sigma_idx"]

    fixed_pts = original.loc[original.index.intersection(list(fixed_idx))]
    sigma_pts = original.loc[original.index.intersection(list(sigma_idx))]

    mu_p0 = result["mu_p0"]
    sd_p0 = result["sd_p0"]
    mu_s0 = result["mu_s0"]
    sd_s0 = result["sd_s0"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1) Original + eliminados
    axes[0].scatter(original["superficie_construida_m2"], original["precio"], s=14, alpha=0.2, color="gray", label="Original")
    axes[0].scatter(fixed_pts["superficie_construida_m2"], fixed_pts["precio"], s=26, alpha=0.9, color="orange", label="Elim. umbral fijo")
    axes[0].scatter(sigma_pts["superficie_construida_m2"], sigma_pts["precio"], s=26, alpha=0.9, color="red", label="Elim. 3 sigma")
    axes[0].axvline(surf_thr, color="black", linestyle="--", linewidth=1)
    axes[0].axhline(price_thr, color="black", linestyle="--", linewidth=1)
    axes[0].set_title(f"{label} - Original y eliminados")
    axes[0].set_xlabel("Superficie construida (m2)")
    axes[0].set_ylabel("Precio")
    axes[0].legend(loc="best", fontsize=8)

    # 2) Vista 3 sigma (sobre precio y superficie)
    axes[1].scatter(after_fixed["superficie_construida_m2"], after_fixed["precio"], s=14, alpha=0.25, color="steelblue", label="Tras umbral fijo")
    axes[1].scatter(sigma_pts["superficie_construida_m2"], sigma_pts["precio"], s=26, alpha=0.95, color="red", label="Elim. 3 sigma")
    axes[1].axvline(mu_s0 - 3 * sd_s0, color="purple", linestyle="--", linewidth=1)
    axes[1].axvline(mu_s0 + 3 * sd_s0, color="purple", linestyle="--", linewidth=1, label="Limites 3σ superficie")
    axes[1].axhline(mu_p0 - 3 * sd_p0, color="green", linestyle="--", linewidth=1)
    axes[1].axhline(mu_p0 + 3 * sd_p0, color="green", linestyle="--", linewidth=1, label="Limites 3σ precio")
    axes[1].set_title(f"{label} - Criterio 3σ (precio y superficie)")
    axes[1].set_xlabel("Superficie construida (m2)")
    axes[1].set_ylabel("Precio")
    axes[1].legend(loc="best", fontsize=8)

    # 3) Despues
    axes[2].scatter(clean["superficie_construida_m2"], clean["precio"], s=16, alpha=0.35, color="tab:blue")
    axes[2].set_title(f"{label} - Despues de limpieza")
    axes[2].set_xlabel("Superficie construida (m2)")
    axes[2].set_ylabel("Precio")

    plt.tight_layout()
    plt.show()


plot_outliers_clear(sale_res, "SALE", price_thr=1_400_000, surf_thr=750)
plot_outliers_clear(rent_res, "RENT", price_thr=3_900, surf_thr=300)

In [ ]:
sale_clean = sale_res["clean"].copy()
rent_clean = rent_res["clean"].copy()

sale_clean.to_csv(SALE_OUT_PATH, index=False)
rent_clean.to_csv(RENT_OUT_PATH, index=False)

sale_check = pd.read_csv(SALE_OUT_PATH)
rent_check = pd.read_csv(RENT_OUT_PATH)

sale_left = int(((sale_check["precio"] > 1_400_000) | (sale_check["superficie_construida_m2"] > 750)).sum())
rent_left = int(((rent_check["precio"] > 3_900) | (rent_check["superficie_construida_m2"] > 300)).sum())

print(f"CSV exportado: {SALE_OUT_PATH}")
print(f"CSV exportado: {RENT_OUT_PATH}")
print(f"Filas SALE final: {sale_check.shape[0]}")
print(f"Filas RENT final: {rent_check.shape[0]}")
print(f"Regla fija restante SALE: {sale_left}")
print(f"Regla fija restante RENT: {rent_left}")

assert sale_left == 0
assert rent_left == 0
print("Verificacion final OK.")